# Multivariate weather -- structural diagnostics on one channel's predictive covariance

Diagnostic notebook (no paper figure; PNGs are written only with `SAVE_FIGS = True`).

Two PatchTST runs on **weather, multivariate (`features=M`, 21 channels),
`seq_len=336`, `pred_len=96`** (the cadence is 10 min, so the horizon is 16 h
and the input window 56 h). Both parameterize the predictive distribution with
SCORE (two-stage fit, DFT core) and differ only in the loss stage 1
fits the physical marginals with:

| arm | covariance | fitted with |
|---|---|---|
| `SCORE (stage-1 CRPS)` | $\Sigma = D\,F^{H}\mathrm{diag}(s)\,F\,D$ per channel | CRPS (marginals) then kernel score (core) |
| `SCORE (stage-1 kernel)` | as above | marginal kernel score, then kernel score (core) |

Everything else -- trunk, head sizing, core activation (`softmax`,
$\mathrm{mean}(s)=1$), basis (native DFT), `rank=0`, stage-2 loss -- is
identical, so any difference below is attributable to what stage 1 optimized.

**Why a single channel.** The event here is $(C, T) = (21, 96)$, so
$\Sigma$ is $2016\times2016$ and not plottable. It does not need to be: the
core's FFT runs over the *spatial* (= lead-time) axes only, so
$\Sigma$ is **block-diagonal across channels** and the cross-channel entries
are exactly zero (see `SpectralDiag._circulant_cov_block`). Restricting to one
channel therefore throws away nothing this parameterization can express -- it
is a marginal of a Gaussian, hence Gaussian, with covariance the corresponding
$96\times96$ block -- and every diagnostic from the univariate ETTh1 notebook
carries over verbatim. Section 3 checks the block-diagonality numerically
rather than trusting the argument, because that check is also what validates
the flat-index convention (`index = channel * pred_len + lead`) the block
extraction assumes.

The notebook is organised as:

1. configuration (hardcoded checkpoint paths, channel choice),
2. rebuilding each run from its logged Hydra config + checkpoint,
3. the channel restriction and what we extract from it,
4. **the main figure**: one row per method, columns = forecast /
   autocorrelation w.r.t. the first predicted step / covariance heatmap,
5. **further diagnostics** that separate the parameterizations in ways the main
   figure cannot (see the overview in section 6).

## 0 Setup

In [ ]:
import math
import os
import sys
from pathlib import Path

# --- Project-root bootstrap: put the repo root on sys.path so `import src...`
# works, and export PROJECT_ROOT for any Hydra ${oc.env:PROJECT_ROOT} lookups. ---
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".project-root").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("PROJECT_ROOT", str(PROJECT_ROOT))

import hydra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.lines import Line2D
from omegaconf import OmegaConf
from src.utils.legacy import upgrade_legacy_targets
from scipy import stats as sps

from src.evaluation.evaluator import Evaluator
from src.methods import PredictiveMethod
from src.methods.base import _as_list
from src.models.gaussian_predictive import LightningPredictiveModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(sys.executable)
print("project root:", PROJECT_ROOT, "| device:", DEVICE)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
})

## 1 Configuration

Checkpoint paths are hardcoded run directories under `logs/train/runs/`.
Each entry is a **run directory**, not a `.ckpt`: the
`.hydra/config.yaml` inside it is what lets us rebuild the exact method +
backbone. `find_checkpoint` then prefers `checkpoints/stage2/*.ckpt` (SCORE
runs) over `checkpoints/*.ckpt`.

`CHANNEL` is the channel every panel below is restricted to. It accepts a
column name from the CSV or an integer index into the dataset's canonical
column order (all columns except the target, then the target last -- so
`-1` is always the benchmark's target column `OT`). Nothing else in the
notebook is channel-specific, so re-running with a different `CHANNEL`
reproduces the whole analysis for another variable.

In [ ]:
RUNS = {
    # experiments=train/time_series/multivariate/weather_score
    #   method=score  method.rank=0  method.stage1_loss=crps
    "SCORE (stage-1 CRPS)":
        PROJECT_ROOT / "logs/train/runs/2026-08-12_12-43-37",
    # ... same, stage1_loss left at the config default (marginal_kernel_score)
    "SCORE (stage-1 kernel)":
        PROJECT_ROOT / "logs/train/runs/2026-08-12_13-47-56",
    # A Diag reference can be added here; the run below is a short smoke run
    # without evaluation, so it is left out rather than shown next to two
    # converged arms. Everything below is generic over `RUNS`: uncommenting
    # it adds a row wherever a row makes sense.
    # "Diag (CRPS)": PROJECT_ROOT / "logs/train/runs/2026-08-12_13-33-51",
}

COLORS = {
    "SCORE (stage-1 CRPS)": "#0F7BA2",
    "SCORE (stage-1 kernel)": "#D1495B",
    "Diag (CRPS)": "#4C4C4C",
}

CHANNEL = 'OT'         # column name, or an int index (-1 = the target column)
SAMPLE_INDEX = 400      # which test window the single-sample panels show
N_SAMPLES = 200         # ensemble members drawn for the sample-path panel
DENORMALIZE = True      # undo the dataset standardisation -> physical units
CONTEXT_SHOW = None     # None = the full 336-step input window; or an int
MAX_TEST_BATCHES = None # None = the whole test split; an int truncates (debug)
EVAL_BATCH_SIZE = 128

SAVE_FIGS = False
FIG_DIR = PROJECT_ROOT / "notebooks" / "figures"


def savefig(fig, name):
    """Write `fig` to notebooks/figures/<name>.png when SAVE_FIGS is on."""
    if not SAVE_FIGS:
        return
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")

## 2 Rebuilding a run from its checkpoint

The checkpoint stores only `backbone.*` weights -- the predictive method is not
an `nn.Module` and is not serialized -- so both have to be rebuilt from the
logged config exactly as `src/train/train.py` does it. Two things matter and are
easy to get wrong:

* the bandwidths are **derived at runtime** (`gamma = sqrt(gamma_* * d)`,
  `d = C_out * pred_len`, here $d = 21 \cdot 96 = 2016$), not stored in the
  method config;
* `PatchTST` needs `group_denorm` and `other_zero_init` from the method, because
  its RevIN layer denormalizes each head group by its own rule (`loc` / `var` /
  `scale` / `none`). For SCORE the `other` group (the core
  spectrum) is denorm-`none` while `mu`/`diag` follow the marginals, so omitting
  it silently rescales the core.

In [ ]:
_DM_CACHE, _MODEL_CACHE = {}, {}


def load_run_config(run_dir):
    """The fully-composed Hydra config logged with a run."""
    cfg_path = Path(run_dir) / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(f"No logged config at {cfg_path}")
    return upgrade_legacy_targets(OmegaConf.load(cfg_path))


def build_datamodule(cfg, num_workers=0, batch_size=EVAL_BATCH_SIZE, use_cache=True):
    """The run's datamodule with the test split set up (no worker processes).

    Cached on the data-shaping fields, so the arms -- which share one dataset --
    read the CSV once.
    """
    dm_cfg = cfg.datamodule
    key = (dm_cfg._target_, dm_cfg.get("data_path"), dm_cfg.get("features"),
           dm_cfg.get("seq_len"), dm_cfg.get("pred_len"), dm_cfg.get("data_split"),
           dm_cfg.get("scale"), batch_size)
    if use_cache and key in _DM_CACHE:
        return _DM_CACHE[key]
    dm = hydra.utils.instantiate(dm_cfg, num_workers=num_workers,
                                 eval_batch_size=batch_size,
                                 persistent_workers=False)
    dm.setup("test")
    if use_cache:
        _DM_CACHE[key] = dm
    return dm


def find_checkpoint(run_dir):
    """The deployed checkpoint: `checkpoints/stage2/*.ckpt` if staged, else `checkpoints/*.ckpt`."""
    ck = Path(run_dir) / "checkpoints"
    found = sorted(ck.glob("stage2/*.ckpt")) or sorted(ck.glob("*.ckpt"))
    if not found:
        raise FileNotFoundError(f"No .ckpt under {ck}")
    return str(found[-1])


def build_model_from_run(run_dir, device=DEVICE, use_cache=True):
    """Rebuild (model, method, datamodule, cfg, ckpt_path) for a logged run.

    The staged (whitened) checkpoint is loaded into a plain
    `LightningPredictiveModel`: stage awareness only affects *training*, and the
    state dict is `backbone.*` either way.
    """
    if use_cache and str(run_dir) in _MODEL_CACHE:
        return _MODEL_CACHE[str(run_dir)]
    cfg = load_run_config(run_dir)
    dm = build_datamodule(cfg)

    d = dm.get_total_dimension()
    gamma_train = [math.sqrt(f * d) for f in _as_list(cfg.get("gamma_train", [1.0]))]
    gamma_eval = [math.sqrt(f * d) for f in _as_list(cfg.get("gamma_eval", [1.0]))]

    method: PredictiveMethod = hydra.utils.instantiate(
        cfg.method, out_channels=dm.get_channels(),
        gamma=gamma_train, gamma_eval=gamma_eval)
    backbone = hydra.utils.instantiate(
        cfg.backbone, in_channels=dm.get_input_channels(),
        out_channels=method.out_channels, group_widths=method.group_widths,
        group_denorm=method.group_denorm, other_zero_init=method.other_zero_init)

    ckpt = find_checkpoint(run_dir)
    state = torch.load(ckpt, map_location="cpu")["state_dict"]
    model = LightningPredictiveModel(backbone=backbone, predictive_method=method)
    model.load_state_dict(state)
    model.eval().to(device)
    method.training = False  # eval mode -> gamma_eval / n_eval_samples

    out = (model, method, dm, cfg, ckpt)
    if use_cache:
        _MODEL_CACHE[str(run_dir)] = out
    return out

## 3 What we extract

Every family keeps its covariance implicit (a $2016^2$ matrix per window is
16 MiB, and 4.6 GiB at ERA5 scale), so there is no `get_cov`. But we only ever
need the rows belonging to **one channel**, and those we can materialize from
the two accessors every family implements, using the identity the dense masked
scores already rely on (`PredictiveMethod._dense_score_terms`):

$$\Sigma_{ij} = \tfrac12\bigl(\Sigma_{ii} + \Sigma_{jj} - \mathrm{Var}(X_i - X_j)\bigr),$$

with $\Sigma_{ii}$ from `get_marg_std` and $\mathrm{Var}(X_i-X_j)$ from
`pair_diff_var`, which takes exactly this kind of row block. It is exact and
**family-agnostic** -- it goes through the same closed forms the scores use, so
the plots below cannot drift from what is being optimized. (Note it recovers the
*physical* $\Sigma$ even for SCORE, whose own `_dense_score_terms` override
returns the correlation core of the standardized target instead.)

The row block is $(B, 96, 2016)$: the $96\times96$ block on the chosen channel
is what every panel uses, and the remaining $96 \times 1920$ columns are the
cross-channel part, which `block_covariance` returns the max modulus of. That
number being exactly 0 is both the check that this parameterization has no
cross-channel dependence and the check that the flat index is
`channel * pred_len + lead` rather than the other way round -- a transposed
convention would put a nonzero block there.

The event axes are also what fixes the time base: the CSV is 10-minute data, so
lead $h$ is $h/6$ hours; `step_hours` reads that off the file rather than
assuming it, and every axis below is in **hours**.

Everything is reported in the channel's own physical units by undoing the
dataset's train-range standardisation: a mean shifts and scales, a std scales,
a covariance scales by the square.

In [ ]:
def read_csv_header(path):
    """The CSV's column names, with a latin-1 fallback.

    The weather file's unit strings (`SWDR (W/m2)`) carry non-ASCII bytes, so
    the fallback keeps this working on files that were not written as utf-8.
    """
    for enc in ("utf-8", "latin-1"):
        try:
            return list(pd.read_csv(path, nrows=0, encoding=enc).columns)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"cannot decode the header of {path}")


def channel_names(cfg):
    """Value columns in the dataset's canonical order: others, then the target.

    `TimeSeriesDataset.__read_data__` reorders to `['date'] + others + [target]`,
    so the target is the last channel and `scaler.mean_/scale_` follow the same
    order.
    """
    cols = [c for c in read_csv_header(cfg.datamodule.data_path) if c != "date"]
    target = cfg.datamodule.get("target", "OT")
    if cfg.datamodule.get("features") == "S":
        return [target]
    cols.remove(target)
    return cols + [target]


def resolve_channel(cfg, channel=CHANNEL):
    """(index, name) of the channel to restrict to, from a name or an index."""
    names = channel_names(cfg)
    if isinstance(channel, str):
        if channel not in names:
            raise KeyError(f"{channel!r} is not a column of this dataset: {names}")
        return names.index(channel), channel
    idx = int(channel) % len(names)
    return idx, names[idx]


def step_hours(cfg):
    """Sampling interval of the series, in hours (weather: 10 min -> 1/6 h)."""
    dates = pd.read_csv(cfg.datamodule.data_path, usecols=["date"], nrows=2)["date"]
    delta = pd.to_datetime(dates.iloc[1]) - pd.to_datetime(dates.iloc[0])
    return delta.total_seconds() / 3600.0


def channel_rows(channel_idx, pred_len, device):
    """Flat event indices of one channel: `channel * pred_len + lead`."""
    return torch.arange(pred_len, device=device) + channel_idx * pred_len


def block_covariance(method, pred, rows):
    """(B, R, R) predictive covariance on `rows`, plus the off-block max modulus.

    The second return value is `max |Sigma_ij|` over the columns *outside*
    `rows`, i.e. the largest cross-channel coupling this window's predictive
    assigns to the selected channel. It costs nothing extra -- the row block
    spans all V columns anyway -- and is what section 3 claims is zero.
    """
    var = method.get_marg_std(pred).flatten(start_dim=1).pow(2)      # (B, V)
    pair = method.pair_diff_var(pred, rows)                          # (B, R, V)
    rows_cov = 0.5 * (var[:, rows].unsqueeze(-1) + var.unsqueeze(-2) - pair)

    off = torch.zeros((), device=var.device, dtype=var.dtype)
    if rows.numel() < var.shape[1]:
        keep = torch.ones(var.shape[1], dtype=torch.bool, device=var.device)
        keep[rows] = False
        off = rows_cov[..., keep].abs().amax()

    cov = rows_cov[..., rows]
    return 0.5 * (cov + cov.transpose(-1, -2)), off  # symmetrize away rounding


def cov_to_corr(cov, eps=1e-12):
    """Correlation matrix; NaN wherever a marginal variance is zero (Dirac)."""
    d = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    denom = np.outer(d, d)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(denom > eps, cov / np.where(denom > eps, denom, 1.0), np.nan)


def has_spread(entry):
    """False for a degenerate (Dirac) predictive -- nothing to correlate."""
    key = "std" if "std" in entry else "sigma"
    return bool(np.all(np.asarray(entry[key]) > 0))


def transform_kind(cfg):
    """The core's basis: 'fourier' for the native FFT path, else the configured kind."""
    tf = cfg.method.get("transform") if "transform" in cfg.method else None
    if tf is None:
        return "fourier"
    kind = tf.get("kind", "custom")
    return kind if isinstance(kind, str) else "-".join(str(k) for k in kind)


def grid_axes(n, ncols=3, panel=(3.5, 3.5)):
    """A wrapped subplot grid for n per-method panels, with the spares removed."""
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(panel[0] * ncols, panel[1] * nrows),
                             layout="constrained", squeeze=False)
    flat = axes.ravel()
    for ax in flat[n:]:
        ax.remove()
    return fig, list(flat[:n])

In [ ]:
for name, rd in RUNS.items():
    break
model, method, dm, cfg, ckpt = build_model_from_run(RUNS[name])
channel_names(cfg)

In [ ]:
@torch.no_grad()
def analyze_sample(name, run_dir, channel=CHANNEL, sample_index=SAMPLE_INDEX,
                   n_samples=N_SAMPLES, denormalize=DENORMALIZE, device=DEVICE):
    """One test window on one channel: context, truth, mean, std, Sigma, draws.

    `raw` keeps the processed head output for that window *and channel* --
    (pred_len, G) with G the method's group axis ([mu, diag, s] for the whitened
    core, [mu, diag] for Diag, [mu, diag, U...] for LorD) -- so section
    6.8 can look at the learned parameters directly. It is None for a
    deterministic head, which has no group axis.
    """
    model, method, dm, cfg, ckpt = build_model_from_run(run_dir, device=device)
    ds = dm.test_dataset
    c, cname = resolve_channel(cfg, channel)
    x, y = ds[sample_index]
    xb = x.unsqueeze(0).to(device)

    pred = model(xb)
    rows = channel_rows(c, y.shape[-1], device)
    mean = method.get_mean(pred)
    std = method.get_marg_std(pred)
    cov, off = block_covariance(method, pred, rows)
    samples = method.get_samples(pred, n_samples=n_samples)

    # Undo the standardisation: locations shift+scale, scales scale, cov scales^2.
    scale = float(ds.scaler.scale_[c]) if (denormalize and ds.scale) else 1.0
    shift = float(ds.scaler.mean_[c]) if (denormalize and ds.scale) else 0.0

    return {
        "name": name,
        "run_dir": str(run_dir),
        "ckpt": ckpt,
        "method_cls": cfg.method._target_.split(".")[-1],
        "loss": cfg.method.get("stage2_loss") or cfg.method.get("loss"),
        "stage1_loss": cfg.method.get("stage1_loss"),
        "channel": c,
        "channel_name": cname,
        "n_channels": int(y.shape[0]),
        "dt": step_hours(cfg),
        "context": x[c].cpu().numpy() * scale + shift,       # (seq_len,)
        "target": y[c].cpu().numpy() * scale + shift,        # (pred_len,)
        "mean": mean[0, c].cpu().numpy() * scale + shift,    # (pred_len,)
        "std": std[0, c].cpu().numpy() * scale,              # (pred_len,)
        "cov": cov[0].cpu().numpy() * scale**2,              # (pred_len, pred_len)
        "cross_max": float(off) * scale**2,                  # max |Sigma| off-block
        "samples": samples[0, c].cpu().numpy() * scale + shift,  # (pred_len, M)
        "raw": pred[0, c].cpu().numpy() if pred.dim() == 4 else None,
        "transform": transform_kind(cfg),
        "units": cname if scale != 1.0 else "norm.",
        "scale": scale,
    }

In [ ]:
@torch.no_grad()
def scan_test_set(name, run_dir, channel=CHANNEL, max_batches=MAX_TEST_BATCHES,
                  device=DEVICE, denormalize=DENORMALIZE):
    """One pass over the test split, restricted to `channel`.

    Returns per-window residuals and marginal stds on that channel, the running
    aggregate variances Var(sum_{h<=H} r_h) implied by its Sigma block, the mean
    predictive block, the empirical residual covariance, the Mahalanobis
    statistic r^T Sigma^-1 r of the block (None for a degenerate predictive), and
    the largest cross-channel |Sigma_ij| seen anywhere in the split.

    The test loader keeps `drop_last=True` (the training protocol), so the last
    partial batch of windows is not scored here either.
    """
    model, method, dm, cfg, ckpt = build_model_from_run(run_dir, device=device)
    ds = dm.test_dataset
    c, cname = resolve_channel(cfg, channel)
    scale = float(ds.scaler.scale_[c]) if (denormalize and ds.scale) else 1.0

    resid, sigmas, aggvar, maha = [], [], [], []
    cov_sum, n, cross = None, 0, 0.0
    rows = None
    for bi, (x, y) in enumerate(dm.test_dataloader()):
        if max_batches is not None and bi >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        if rows is None:
            rows = channel_rows(c, y.shape[-1], device)
        pred = model(x)
        mu = method.get_mean(pred)[:, c]
        sd = method.get_marg_std(pred)[:, c]
        cov, off = block_covariance(method, pred, rows)
        r = y[:, c] - mu

        cross = max(cross, float(off))
        cov_sum = cov.sum(0) if cov_sum is None else cov_sum + cov.sum(0)
        n += r.shape[0]
        resid.append(r.cpu())
        sigmas.append(sd.cpu())
        # Var(sum_{h<=H}) for every H at once: the double cumulative sum of the
        # block, read off its diagonal.
        c2 = cov.cumsum(-1).cumsum(-2)
        aggvar.append(torch.diagonal(c2, dim1=-2, dim2=-1).cpu())

        if bool((sd > 0).all()):
            jitter = 1e-8 * torch.diagonal(cov, dim1=-2, dim2=-1).mean(-1)
            eye = torch.eye(cov.shape[-1], device=cov.device, dtype=cov.dtype)
            chol = torch.linalg.cholesky(cov + jitter[:, None, None] * eye)
            sol = torch.cholesky_solve(r.unsqueeze(-1), chol).squeeze(-1)
            maha.append((r * sol).sum(-1).cpu())

    resid = torch.cat(resid).numpy() * scale
    sigmas = torch.cat(sigmas).numpy() * scale
    return {
        "name": name,
        "n": n,
        "channel": c,
        "channel_name": cname,
        "dt": step_hours(cfg),
        "resid": resid,                                    # (N, pred_len)
        "sigma": sigmas,                                   # (N, pred_len)
        "agg_var": torch.cat(aggvar).numpy() * scale**2,   # (N, pred_len)
        "mean_cov": (cov_sum / n).cpu().numpy() * scale**2,
        "emp_cov": np.cov(resid, rowvar=False),
        "maha": torch.cat(maha).numpy() if maha else None,
        "cross_max": cross * scale**2,
        "units": cname if scale != 1.0 else "norm.",
        "scale": scale,
    }

## 4 Load the runs

The single-window pass first: it also reports the channel, the physical unit
the panels are in, and the cross-channel maximum discussed in section 3.

In [ ]:
results = {name: analyze_sample(name, rd) for name, rd in RUNS.items()}

for r in results.values():
    print(f"{r['name']:36s} {r['method_cls']:22s} basis={r['transform']}")
    print(f"{'':36s} stage1={str(r['stage1_loss']):22s} stage2/loss={r['loss']}")
    print(f"{'':36s} ckpt   {Path(r['ckpt']).relative_to(PROJECT_ROOT)}")
    print(f"{'':36s} channel {r['channel']} = {r['channel_name']!r} "
          f"of {r['n_channels']}, dt = {r['dt'] * 60:.0f} min")
    print(f"{'':36s} sample {SAMPLE_INDEX}: mean sigma = {r['std'].mean():.4f}, "
          f"Sigma block is {r['cov'].shape}, "
          f"max |Sigma| off-block = {r['cross_max']:.3e}")

In [ ]:
scans = {name: scan_test_set(name, rd) for name, rd in RUNS.items()}

for s in scans.values():
    maha = "-- (Dirac)" if s["maha"] is None else f"{s['maha'].mean():.1f}"
    print(f"{s['name']:36s} n={s['n']:5d}  "
          f"RMSE={np.sqrt((s['resid']**2).mean()):.4f}  "
          f"mean sigma={s['sigma'].mean():.4f}  "
          f"mean r'Sigma^-1 r={maha}  (d={s['resid'].shape[1]})  "
          f"max |Sigma| off-block={s['cross_max']:.3e}")

## 5 Main figure

One row per method; columns:

1. **forecast** -- the 336-step input window, the true 96-step future, the
   predictive mean and the $\pm2\sigma$ *marginal* band, all on the selected
   channel;
2. **autocorrelation w.r.t. the first predicted step** -- row 1 of the
   correlation matrix, $\mathrm{corr}(h_1, h)$, with the empirical residual
   correlation of the same arm over the test split as a dashed reference;
3. **the full $96\times96$ covariance block** as a `pcolormesh`.

The covariance colour scale is shared across rows and set by the largest
**off-diagonal** entry, so the off-diagonal structure stays visible; the
diagonal (the marginal variances, an order of magnitude larger) saturates by
design.

In [ ]:
def _cov_vmax(results):
    """Shared symmetric colour limit from the largest off-diagonal |Sigma_ij|."""
    vals = []
    for r in results.values():
        c = np.abs(r["cov"]).copy()
        np.fill_diagonal(c, 0.0)
        vals.append(float(c.max()))
    v = max(vals)
    return v if v > 0 else 1.0


def plot_main_grid(results, scans=None, context_show=CONTEXT_SHOW):
    names = list(results)
    ref = results[names[0]]
    pred_len, seq_len, dt = ref["target"].shape[0], ref["context"].shape[0], ref["dt"]
    show = seq_len if context_show is None else min(context_show, seq_len)
    t_ctx = np.arange(-show, 0) * dt
    t_fut = np.arange(1, pred_len + 1) * dt
    vmax = _cov_vmax(results)

    fig, axes = plt.subplots(len(names), 3, figsize=(13.5, 3.15 * len(names)),
                             gridspec_kw={"width_ratios": [1.9, 1.0, 1.15]},
                             layout="constrained")
    axes = np.atleast_2d(axes)
    for i, name in enumerate(names):
        r = results[name]
        c = COLORS.get(name, f"C{i}")

        # -- forecast -------------------------------------------------------
        ax = axes[i, 0]
        ax.plot(t_ctx, r["context"][-show:], color="0.45", lw=0.8,
                label="input (past states)")
        ax.plot(t_fut, r["target"], color="k", lw=1.2, label="truth")
        ax.plot(t_fut, r["mean"], color=c, lw=1.3, label="predictive mean")
        ax.fill_between(t_fut, r["mean"] - 2 * r["std"], r["mean"] + 2 * r["std"],
                        color=c, alpha=0.22, lw=0, label=r"$\pm2\sigma$ (marginal)")
        ax.axvline(0.0, color="0.6", lw=0.8, ls=":")
        ax.set_ylabel(f"{name}\n[{r['units']}]")
        ax.set_xlabel("time relative to forecast origin [h]")
        ax.legend(loc="upper left", ncols=2, fontsize=7, framealpha=0.9)
        if i == 0:
            ax.set_title("forecast: input, truth, mean, marginal band")

        # -- autocorrelation w.r.t. the first predicted step -----------------
        ax = axes[i, 1]
        corr = cov_to_corr(r["cov"])
        if has_spread(r):
            ax.plot(t_fut, corr[0], color=c, lw=1.4, label="model")
        else:
            ax.text(0.5, 0.55, r"$\Sigma \equiv 0$" "\n(no correlation defined)",
                    transform=ax.transAxes, ha="center", va="center", fontsize=8,
                    color="0.35")
        if scans is not None:
            emp = cov_to_corr(scans[name]["emp_cov"])
            ax.plot(t_fut, emp[0], color="0.35", lw=1.0, ls="--",
                    label="empirical residuals (test)")
        ax.axhline(0.0, color="0.6", lw=0.8)
        ax.set_ylim(-0.35, 1.05)
        ax.set_xlabel("lead time [h]")
        ax.set_ylabel(r"corr$(h_1,\, h)$")
        ax.legend(loc="upper right", fontsize=7)
        if i == 0:
            ax.set_title("autocorrelation w.r.t. first predicted step")

        # -- covariance block -------------------------------------------------
        ax = axes[i, 2]
        im = ax.pcolormesh(t_fut, t_fut, r["cov"], cmap="RdBu_r",
                           vmin=-vmax, vmax=vmax, shading="nearest",
                           rasterized=True)
        ax.set_aspect("equal")
        ax.set_xlabel("lead time [h]")
        ax.set_ylabel("lead time [h]")
        if not has_spread(r):
            ax.text(0.5, 0.5, r"$\Sigma \equiv 0$", transform=ax.transAxes,
                    ha="center", va="center", fontsize=10, color="0.35")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03,
                     label=f"cov [{r['units']}$^2$]")
        if i == 0:
            ax.set_title(r"predictive covariance $\Sigma$ on this channel"
                         "\n(scale set by off-diagonal; diagonal saturates)")

    fig.suptitle(f"weather multivariate ({ref['n_channels']} channels), "
                 f"channel {ref['channel_name']!r}, test window {SAMPLE_INDEX} "
                 f"(pred_len = {pred_len} = {pred_len * dt:.0f} h)", fontsize=11)
    return fig


fig = plot_main_grid(results, scans)
savefig(fig, "ts_multivariate_main")
plt.show()

## 6 Further diagnostics

The main figure shows one window and one row of the correlation matrix. The
panels below target the structural questions it leaves open, each answering a
different one:

| § | diagnostic | question it answers |
|---|---|---|
| 6.1 | correlation matrices vs empirical | is the *shape* of the dependence right, independent of the variance profile? |
| 6.2 | lag-averaged ACF + several anchor rows | is the learned dependence stationary, and does the circulant core wrap around? |
| 6.3 | spread vs skill per lead | are the marginals calibrated as a function of lead time? |
| 6.4 | eigenspectrum of $\Sigma$ | how many directions of variability can the parameterization actually express? |
| 6.5 | variance of the running aggregate | does the dependence give the right uncertainty for *window functionals* (the thing off-diagonals are for)? |
| 6.6 | PIT + $r^\top\Sigma^{-1}r$ vs $\chi^2_d$ | marginal and joint calibration, on this channel |
| 6.7 | ensemble sample paths | do draws look like plausible trajectories, or white noise around the mean? |
| 6.8 | the learned parameters themselves | *what* did the core learn on this channel? |
| 6.9 | test-set scores | how the structure translates into the reported scores |

All of them are the univariate notebook's diagnostics evaluated on the
$96\times96$ block, which is legitimate precisely because a marginal of a
Gaussian is a Gaussian with the corresponding sub-covariance. The one thing
they cannot see is cross-channel dependence -- and section 4 has already
reported that there is none to see.

Arms with a degenerate predictive ($\Sigma \equiv 0$) are skipped wherever a
correlation, an eigenvalue or a calibration statistic is undefined.

### 6.1 Correlation matrices, model vs empirical

The covariance heatmap in the main figure is dominated by the marginal variance
profile. Dividing it out isolates the *dependence structure*, which is the part
the parameterization is responsible for. The right-hand panel is the empirical
correlation of the residuals over the test split -- the target these should
match (exactly so if the mean is unbiased and the error process is stationary
across windows).

The average predictive covariance over the test split is used here rather than a
single window, since SCORE's $D$ is input-dependent.

In [ ]:
def plot_correlation_panels(scans, ncols=3):
    names = list(scans)
    ref = scans[names[0]]
    H, dt = ref["mean_cov"].shape[0], ref["dt"]
    t = np.arange(1, H + 1) * dt

    fig, axes = grid_axes(len(names) + 1, ncols=ncols, panel=(3.5, 3.6))
    for i, (ax, name) in enumerate(zip(axes, names)):
        s = scans[name]
        corr = cov_to_corr(s["mean_cov"])
        im = ax.pcolormesh(t, t, corr, cmap="RdBu_r", vmin=-1, vmax=1,
                           shading="nearest", rasterized=True)
        if not has_spread(s):
            ax.text(0.5, 0.5, r"$\Sigma \equiv 0$", transform=ax.transAxes,
                    ha="center", va="center", fontsize=10, color="0.35")
        ax.set_aspect("equal")
        ax.set_title(f"{name}\nmodel correlation")
        ax.set_xlabel("lead [h]")
        ax.set_ylabel("lead [h]")

    ax = axes[-1]
    ref_name = next(n for n in reversed(names) if has_spread(scans[n]))
    im = ax.pcolormesh(t, t, cov_to_corr(scans[ref_name]["emp_cov"]), cmap="RdBu_r",
                       vmin=-1, vmax=1, shading="nearest", rasterized=True)
    ax.set_aspect("equal")
    ax.set_title(f"empirical residual correlation\n(test, {ref_name})")
    ax.set_xlabel("lead [h]")
    ax.set_ylabel("lead [h]")
    fig.colorbar(im, ax=axes, fraction=0.025, pad=0.01, label="correlation")
    return fig


fig = plot_correlation_panels(scans)
savefig(fig, "ts_multivariate_correlation")
plt.show()

### 6.2 Autocorrelation: stationarity and wrap-around

Left: the **lag-averaged** ACF -- the mean of each diagonal of the correlation
matrix. For a stationary (Toeplitz) dependence this loses nothing; the gap
between it and the individual rows on the right measures how non-stationary the
learned structure is.

Right: several rows of the correlation matrix, anchored at different leads. This
is where the *circulant* core shows itself: $F^H\mathrm{diag}(s)F$ is periodic
with period `pred_len`, so correlation that should decay to zero instead comes
back up as $h \to 96$ and every anchor row is a shifted copy of the same
profile. Whether that is harmless or a real misfit is exactly what the empirical
reference in 6.1 decides.

Anchors are given in *steps*; at 10-minute cadence the 96-step horizon is 16 h,
so a lag of 48 steps is 8 h.

In [ ]:
def lag_acf(corr):
    """Diagonal-averaged autocorrelation: the stationary part of `corr`."""
    return np.array([np.nanmean(np.diagonal(corr, offset=k))
                     for k in range(corr.shape[0])])


def plot_acf(scans, anchors=(1, 32, 64)):
    names = [n for n in scans if has_spread(scans[n])]
    skipped = [n for n in scans if n not in names]
    ref = scans[list(scans)[0]]
    H, dt = ref["mean_cov"].shape[0], ref["dt"]
    styles = ["-", "--", ":"]

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.7), layout="constrained")

    ax = axes[0]
    for i, name in enumerate(names):
        ax.plot(np.arange(H) * dt, lag_acf(cov_to_corr(scans[name]["mean_cov"])),
                color=COLORS.get(name, f"C{i}"), lw=1.4, label=name)
    ref_name = names[-1]
    ax.plot(np.arange(H) * dt, lag_acf(cov_to_corr(scans[ref_name]["emp_cov"])),
            color="0.3", ls="--", lw=1.1, label=f"empirical ({ref_name})")
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set_xlabel("lag [h]")
    ax.set_ylabel("autocorrelation")
    ax.set_title("lag-averaged ACF")
    ax.legend(fontsize=7)

    ax = axes[1]
    for i, name in enumerate(names):
        corr = cov_to_corr(scans[name]["mean_cov"])
        for j, a in enumerate(anchors):
            ax.plot(np.arange(1, H + 1) * dt, corr[a - 1],
                    color=COLORS.get(name, f"C{i}"), lw=1.1,
                    ls=styles[j % len(styles)])
    ax.axhline(0, color="0.6", lw=0.8)
    ax.set_xlabel("lead [h]")
    ax.set_ylabel("correlation")
    ax.set_title("rows of the correlation matrix (non-stationarity / wrap-around)")
    handles = [Line2D([], [], color=COLORS.get(n, f"C{i}"), lw=1.4, label=n)
               for i, n in enumerate(names)]
    handles += [Line2D([], [], color="0.4", lw=1.1, ls=styles[j % len(styles)],
                       label=f"anchor $h={a}$ ({a * dt:.1f} h)")
                for j, a in enumerate(anchors)]
    ax.legend(handles=handles, fontsize=7, ncols=2)

    if skipped:
        print("skipped (degenerate predictive):", ", ".join(skipped))
    return fig


fig = plot_acf(scans)
savefig(fig, "ts_multivariate_acf")
plt.show()

### 6.3 Spread vs skill per lead time

The marginal half of the problem, which both arms fit in stage 1 -- with
different losses, so this is the panel where the two runs are *supposed* to
differ. The dispersion ratio $\overline{\sigma}(h) / \mathrm{RMSE}(h)$ should
sit at 1; below is under-dispersed, above over-dispersed. A method whose
*shape* is right but whose ratio drifts with lead is telling you the marginals,
not the structure, are the problem.

In [ ]:
def plot_spread_skill(scans):
    names = list(scans)
    ref = scans[names[0]]
    H, dt, units = ref["sigma"].shape[1], ref["dt"], ref["units"]
    t = np.arange(1, H + 1) * dt

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.7), layout="constrained")
    ax = axes[0]
    for i, name in enumerate(names):
        s = scans[name]
        c = COLORS.get(name, f"C{i}")
        ax.plot(t, np.sqrt((s["resid"] ** 2).mean(0)), color=c, lw=1.0, ls="--",
                label=f"{name}: RMSE")
        ax.plot(t, s["sigma"].mean(0), color=c, lw=1.4,
                label=f"{name}: mean $\\sigma$")
    ax.set_xlabel("lead [h]")
    ax.set_ylabel(f"[{units}]")
    ax.set_title("spread vs skill per lead time")
    ax.legend(fontsize=6.5, ncols=2, loc="lower right")

    ax = axes[1]
    for i, name in enumerate(names):
        s = scans[name]
        rmse = np.sqrt((s["resid"] ** 2).mean(0))
        with np.errstate(divide="ignore", invalid="ignore"):
            ax.plot(t, np.where(rmse > 0, s["sigma"].mean(0) / rmse, np.nan),
                    color=COLORS.get(name, f"C{i}"), lw=1.4, label=name)
    ax.axhline(1.0, color="0.4", lw=0.9, ls=":")
    ax.set_ylim(0, 2)
    ax.set_xlabel("lead [h]")
    ax.set_ylabel(r"mean $\sigma$ / RMSE")
    ax.set_title("dispersion ratio (1 = calibrated marginals)")
    ax.legend(fontsize=7)
    return fig


fig = plot_spread_skill(scans)
savefig(fig, "ts_multivariate_spread_skill")
plt.show()

### 6.4 Eigenspectrum of the predictive covariance

How many directions of joint variability the parameterization can express on
this channel, which is a hard structural limit rather than a fitting outcome:
the circulant core has a full spectrum $s$, so it can put mass on many
directions, but only on the *Fourier* ones. The empirical spectrum says how
much of that is actually needed. (A $\mathrm{diag}(v) + uu^\top$ arm, if one is
added to `RUNS`, shows up here as a flat spectrum with a single spike.)

In [ ]:
def plot_eigenspectrum(scans):
    names = [n for n in scans if has_spread(scans[n])]
    units = scans[list(scans)[0]]["units"]
    fig, ax = plt.subplots(figsize=(6.0, 3.8), layout="constrained")
    for i, name in enumerate(names):
        ev = np.linalg.eigvalsh(scans[name]["mean_cov"])[::-1]
        ax.semilogy(np.arange(1, ev.size + 1), np.clip(ev, 1e-12, None),
                    color=COLORS.get(name, f"C{i}"), lw=1.4, label=name)
    ref = names[-1]
    ev = np.linalg.eigvalsh(scans[ref]["emp_cov"])[::-1]
    ax.semilogy(np.arange(1, ev.size + 1), np.clip(ev, 1e-12, None),
                color="0.3", ls="--", lw=1.1, label=f"empirical ({ref})")
    ax.set_xlabel("eigenvalue index")
    ax.set_ylabel(f"eigenvalue [{units}$^2$]")
    ax.set_title("eigenspectrum of the mean predictive covariance")
    ax.legend(fontsize=7)
    return fig


fig = plot_eigenspectrum(scans)
savefig(fig, "ts_multivariate_eigenspectrum")
plt.show()

### 6.5 Variance of the running aggregate

The functional test: off-diagonal covariance exists to get the uncertainty of
*aggregates* right (hourly means over 10-minute data, window totals). For a
window of length $H$,

$$\mathrm{Var}\Bigl(\sum_{h\le H} r_h\Bigr) = \mathbf{1}^\top \Sigma_{1:H,1:H}\, \mathbf{1},$$

which is read straight off the double cumulative sum of the block. Under
independence this grows like $H$; with positive dependence, faster. The
empirical curve is the variance of the corresponding cumulative residual over the
test split, so the ratio in the right panel is a direct answer to "does this
covariance buy anything for window functionals?" -- a diagnostic no marginal
score can give.

In [ ]:
def plot_aggregate_variance(scans):
    names = [n for n in scans if has_spread(scans[n])]
    ref = scans[list(scans)[0]]
    H, dt, units = ref["agg_var"].shape[1], ref["dt"], ref["units"]
    t = np.arange(1, H + 1) * dt

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.7), layout="constrained")
    ax = axes[0]
    for i, name in enumerate(names):
        s = scans[name]
        c = COLORS.get(name, f"C{i}")
        ax.loglog(t, s["agg_var"].mean(0), color=c, lw=1.4, label=f"{name}: model")
        ax.loglog(t, np.cumsum(s["resid"], axis=1).var(0), color=c, lw=1.0,
                  ls="--", label=f"{name}: empirical")
    ax.set_xlabel("window length $H$ [h]")
    ax.set_ylabel(r"Var$\left(\sum_{h\leq H} r_h\right)$" f" [{units}$^2$]")
    ax.set_title("variance of the running aggregate")
    ax.legend(fontsize=6.5)

    ax = axes[1]
    for i, name in enumerate(names):
        s = scans[name]
        emp = np.cumsum(s["resid"], axis=1).var(0)
        with np.errstate(divide="ignore", invalid="ignore"):
            ax.semilogx(t, np.where(emp > 0, s["agg_var"].mean(0) / emp, np.nan),
                        color=COLORS.get(name, f"C{i}"), lw=1.4, label=name)
    ax.axhline(1.0, color="0.4", lw=0.9, ls=":")
    ax.set_ylim(0, 2)
    ax.set_xlabel("window length $H$ [h]")
    ax.set_ylabel("model / empirical")
    ax.set_title("aggregate-variance ratio (1 = correct dependence)")
    ax.legend(fontsize=7)
    return fig


fig = plot_aggregate_variance(scans)
savefig(fig, "ts_multivariate_aggregate_variance")
plt.show()

### 6.6 Calibration, marginal and joint

Left: the PIT $\Phi\bigl((y_h - \mu_h)/\sigma_h\bigr)$ pooled over leads and
windows. Flat = calibrated marginals; a U shape = under-dispersed, a hump =
over-dispersed.

Right: the Mahalanobis statistic $r^\top\Sigma^{-1}r$ of the $96\times96$
block, which for a correctly specified Gaussian is $\chi^2_d$ with $d = 96$.
This is the *joint* version of the same check and the only one here that
penalizes getting the dependence wrong in either direction: too little
correlation inflates the statistic, too much deflates it. Note the two panels
can disagree -- marginals can be fine while the joint is badly off, which is
precisely the regime this project is about.

Because the block is a marginal of the full $2016$-dimensional predictive, this
$\chi^2_{96}$ reference is exact: no approximation is introduced by looking at
one channel.

In [ ]:
def plot_calibration(scans):
    names = [n for n in scans if has_spread(scans[n])]
    d = scans[list(scans)[0]]["sigma"].shape[1]

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.7), layout="constrained")
    ax = axes[0]
    for i, name in enumerate(names):
        s = scans[name]
        pit = sps.norm.cdf((s["resid"] / s["sigma"]).ravel())
        ax.hist(pit, bins=20, range=(0, 1), histtype="step", density=True,
                lw=1.4, color=COLORS.get(name, f"C{i}"), label=name)
    ax.axhline(1.0, color="0.4", lw=0.9, ls=":")
    ax.set_xlabel("PIT")
    ax.set_ylabel("density")
    ax.set_title("marginal PIT (flat = calibrated)")
    ax.legend(fontsize=7)

    ax = axes[1]
    hi = sps.chi2.ppf(0.999, d) * 2.5
    grid = np.linspace(0, hi, 500)
    ax.plot(grid, sps.chi2.pdf(grid, d), color="0.3", ls="--", lw=1.2,
            label=f"$\\chi^2_{{{d}}}$")
    for i, name in enumerate(names):
        ax.hist(scans[name]["maha"], bins=60, histtype="step", density=True,
                lw=1.4, color=COLORS.get(name, f"C{i}"), label=name)
    ax.set_xlim(0, hi)
    ax.set_xlabel(r"$r^\top \Sigma^{-1} r$")
    ax.set_ylabel("density")
    ax.set_title(r"joint calibration vs $\chi^2_d$")
    ax.legend(fontsize=7)
    return fig


fig = plot_calibration(scans)
savefig(fig, "ts_multivariate_calibration")
plt.show()

### 6.7 Ensemble sample paths

The most direct read of what the covariance *means*: the selected channel of
draws from $\mathcal{N}(\mu, \Sigma)$ for one window. A diagonal covariance
produces white noise around the mean; a covariance with a decaying ACF produces
trajectories that look like the series itself. Two models can have identical
marginals and identical CRPS here and still look nothing alike.

In [ ]:
def plot_sample_paths(results, n_show=8, ncols=3):
    names = list(results)
    ref = results[names[0]]
    H, dt = ref["target"].shape[0], ref["dt"]
    t = np.arange(1, H + 1) * dt

    fig, axes = grid_axes(len(names), ncols=ncols, panel=(4.4, 3.5))
    lo = min(r["mean"].min() - 3 * r["std"].max() for r in results.values())
    hi = max(r["mean"].max() + 3 * r["std"].max() for r in results.values())
    for i, (ax, name) in enumerate(zip(axes, names)):
        r = results[name]
        ax.set_ylim(lo, hi)
        c = COLORS.get(name, f"C{i}")
        m = min(n_show, r["samples"].shape[-1])
        if has_spread(r):
            ax.plot(t, r["samples"][:, :m], color=c, lw=0.7, alpha=0.55)
            title = f"{name}\n{m} draws"
        else:
            title = f"{name}\n(Dirac: every draw = the mean)"
        ax.plot(t, r["target"], color="k", lw=1.4, label="truth")
        ax.plot(t, r["mean"], color=c, lw=1.7, label="mean")
        ax.set_title(title)
        ax.set_xlabel("lead [h]")
        ax.set_ylabel(f"[{r['units']}]")
        if i == 0:
            ax.legend(fontsize=7)
    return fig


fig = plot_sample_paths(results)
savefig(fig, "ts_multivariate_sample_paths")
plt.show()

### 6.8 The learned parameters themselves

One level below the covariance: what the structural head emits for this window,
**on this channel**. The core is normalized per channel
($\mathrm{mean}(s) = 1$ over the frequency axis, `softmax` activation), so the
spectra of different channels are separately gauged and the flat line
$s \equiv 1$ is "no structure learned" -- the failure mode where the head
cannot express a per-mode parameter and the core stays pinned near identity.

The bin-to-period map is basis-specific: $96/k$ steps for the DFT (i.e.
$16/k$ hours at this cadence), $192/k$ for the DCT and $192/(k{+}1)$ for the
DST, which resolve half-cycles -- so the spectra are *not* comparable
bin-by-bin across bases (see `spectral_axis`). Note that a 96-step horizon is
16 h here, so unlike the hourly ETTh1 setup there is **no diurnal bin**: the
longest period the core can represent is shorter than a day. The guide line
marks `MARK_PERIOD_H` instead.

In [ ]:
MARK_PERIOD_H = 4.0   # period (in hours) the guide line in 6.8 marks


def spectral_axis(kind, H, dt=1.0):
    """(bins to plot, period of each bin in hours, axis label) for a core basis.

    The bin -> frequency map is basis-specific, so a "period" annotation copied
    between bases would be wrong by a factor of two:

    * `fourier`  exp(2 pi i k j / H)          -> period H / k, and the spectrum
      is conjugate-symmetric, so only k <= H/2 is independent;
    * `dct`      cos(pi k (2j+1) / 2H)        -> period 2H / k, all H bins real
      and independent;
    * `dst`      sin(pi (k+1)(2j+1) / 2H)     -> period 2H / (k+1).

    `dt` converts a period in steps into hours.
    """
    if kind == "fourier":
        k = np.arange(H // 2 + 1)
        with np.errstate(divide="ignore"):
            period = np.where(k > 0, H / np.maximum(k, 1), np.inf)
        return k, period * dt, (r"frequency bin $k$   (period $= $"
                                f"{H * dt:.3g}$/k$ h)")
    if kind == "dst":
        k = np.arange(H)
        return k, 2 * H / (k + 1) * dt, (r"bin $k$   (period $= $"
                                         f"{2 * H * dt:.3g}$/(k{{+}}1)$ h)")
    # dct and any other real-orthogonal cosine-like basis
    k = np.arange(H)
    with np.errstate(divide="ignore"):
        period = np.where(k > 0, 2 * H / np.maximum(k, 1), np.inf)
    return k, period * dt, r"bin $k$   (period $= $" f"{2 * H * dt:.3g}$/k$ h)"


def plot_learned_structure(results, ncols=2, mark_period_h=MARK_PERIOD_H):
    entries = [(n, r) for n, r in results.items() if r["raw"] is not None]
    if not entries:
        print("no structural head to show")
        return None

    fig, axes = grid_axes(len(entries), ncols=ncols, panel=(5.0, 3.5))
    for i, (ax, (name, r)) in enumerate(zip(axes, entries)):
        c = COLORS.get(name, f"C{i}")
        raw, H, dt = r["raw"], r["target"].shape[0], r["dt"]
        if r["method_cls"] == "SCORE":
            k, period, xlabel = spectral_axis(r["transform"], H, dt)
            ax.semilogy(k, raw[k, 2], color=c, lw=1.4, marker="o", ms=2.5)
            ax.axhline(1.0, color="0.4", lw=0.9, ls=":", label=r"$s\equiv1$ (flat)")
            # The bin whose period is closest to `mark_period_h`, in this basis;
            # drawn only if that period is actually resolvable on this horizon.
            finite = np.isfinite(period)
            if finite.any() and period[finite].max() >= mark_period_h:
                km = int(k[finite][np.argmin(np.abs(period[finite] - mark_period_h))])
                ax.axvline(km, color="0.55", lw=0.9, ls="--",
                           label=f"{mark_period_h:g} h period ($k={km}$)")
            ax.set_xlabel(xlabel)
            ax.set_ylabel("core spectrum $s_k$")
            ax.set_title(f"{name}\ncore spectrum, channel {r['channel_name']!r} "
                         f"({r['transform'].upper()} basis)")
            ax.legend(fontsize=7)
        else:  # [mu, diag] (Diag) or [mu, diag, U...] (LorD)
            t = np.arange(1, H + 1) * dt
            ax.plot(t, np.sqrt(raw[:, 1]) * r["scale"], color=c, lw=1.4,
                    label=r"$\sqrt{v}$ (diagonal)")
            for j in range(raw.shape[1] - 2):
                ax.plot(t, raw[:, 2 + j] * r["scale"], color=c, lw=1.1, ls="--",
                        label=f"$u_{{{j + 1}}}$ (low-rank factor)")
            ax.axhline(0.0, color="0.6", lw=0.8)
            ax.set_xlabel("lead [h]")
            ax.set_ylabel(f"[{r['units']}]")
            ax.set_title(f"{name}\nmarginal head, channel {r['channel_name']!r}")
            ax.legend(fontsize=7)
    return fig


fig = plot_learned_structure(results)
if fig is not None:
    savefig(fig, "ts_multivariate_learned_structure")
plt.show()

### 6.9 Scores

Two tables, because "restricted to one channel" changes what a score means:

* **channel-restricted marginal summary** -- computed from the scan above, so
  it is exactly the quantity the panels show: RMSE, mean $\sigma$, the
  dispersion ratio, the closed-form Gaussian CRPS of the marginals, the
  normalized Mahalanobis $\overline{r^\top\Sigma^{-1}r}/d$ (1 = joint
  calibration) and the aggregate-variance ratio over the full window (1 =
  correct dependence);
* **full multivariate scores** -- the repo's own evaluation protocol
  (`src.evaluation.Evaluator` over the metric registry) on the complete
  $21 \times 96$ event, so the numbers are directly comparable with what
  training logs. `mse` / `mae` / `crps` see only the marginals;
  `energy_score`, `kernel_score` and `variogram_score` are the multivariate
  ones, and `variogram_score` is the one that isolates pairwise structure.
  Since $\Sigma$ is block-diagonal across channels, whatever those gain over
  the marginal scores is bought entirely by within-channel lead-time structure.

In [ ]:
def gaussian_crps(resid, sigma):
    """Closed-form CRPS of N(mu, sigma^2) at y, from the residual r = y - mu."""
    z = resid / sigma
    return float(np.mean(sigma * (z * (2 * sps.norm.cdf(z) - 1)
                                  + 2 * sps.norm.pdf(z) - 1 / np.sqrt(np.pi))))


def channel_summary(scans):
    """Per-arm summary of the channel-restricted diagnostics."""
    rows = {}
    for name, s in scans.items():
        d = s["sigma"].shape[1]
        rmse = float(np.sqrt((s["resid"] ** 2).mean()))
        emp_agg = float(np.cumsum(s["resid"], axis=1).var(0)[-1])
        rows[name] = {
            "RMSE": rmse,
            "mean sigma": float(s["sigma"].mean()),
            "sigma/RMSE": float(s["sigma"].mean()) / rmse if rmse > 0 else np.nan,
            "CRPS": gaussian_crps(s["resid"], s["sigma"]),
            "maha/d": float(s["maha"].mean()) / d if s["maha"] is not None else np.nan,
            "agg var ratio": (float(s["agg_var"].mean(0)[-1]) / emp_agg
                              if emp_agg > 0 else np.nan),
        }
    return pd.DataFrame(rows).T


summary = channel_summary(scans)
print(f"channel-restricted ({scans[list(scans)[0]]['channel_name']!r}):")
print(summary.to_string(float_format=lambda v: f"{v:.4f}"))

In [ ]:
def score_table(runs, metrics=("mse", "mae", "crps", "energy_score",
                               "variogram_score", "kernel_score", "nll"),
                device=DEVICE):
    rows = {}
    for name, rd in runs.items():
        model, method, dm, cfg, ckpt = build_model_from_run(rd, device=device)
        ev = Evaluator(method, metrics=list(metrics), device=device,
                       metric_kwargs={"variogram_score": {"weight": "none"}})
        rows[name] = ev.evaluate(model, dm.test_dataloader())
    return pd.DataFrame(rows).T


scores = score_table(RUNS)
print("full multivariate (all 21 channels):")
print(scores.to_string(float_format=lambda v: f"{v:.4f}"))
scores.style.format("{:.4f}")

## 7 What to look for

A checklist for reading the panels above, and for re-reading them after a
retrain:

* **The two arms share everything but stage 1**, so 6.3 (spread vs skill) and
  the left panel of 6.6 are where they are *expected* to differ, and 6.1-6.2
  are where they should not: a visible gap in the correlation shape means the
  stage-1 marginals changed what stage 2 could fit, not just how well the
  marginals themselves are calibrated. That is the single most informative
  comparison this pair supports.
* **CRPS vs marginal kernel score in stage 1** differ in how they weight the
  tails; if 6.3's dispersion ratio separates the arms while 6.6's $\chi^2$
  panel does not, the difference is confined to the marginals and the
  dependence is indifferent to which stage-1 loss was used.
* **The wrap-around** at $h \to 96$ (6.2, right panel) is a property of the
  circulant *basis*, not of the fit. On a 16-hour horizon it is a much weaker
  constraint than on ETTh1's 96 hours, so if it is visible here it is because
  the true ACF has not decayed by 16 h -- check 6.1's empirical panel before
  blaming the basis. Swapping in a DCT/DST run (`method=score_dct`) is the
  controlled test, exactly as in the univariate notebook.
* **6.8 is the first thing to check when a method "doesn't help"**: a spectrum
  pinned near $s \equiv 1$ means the core never learned anything on this
  channel, which is a head capacity/readout problem rather than a modelling
  one, and none of the other panels distinguish that from a genuine "no
  structure to find". Note the `other` head group is *mode*-indexed for this
  method, so a readout without a per-mode parameter would silently pin the
  core near identity.
* **6.3 against 6.6**: calibrated marginals with an off $\chi^2$ is the
  signature of a dependence problem; both off means the marginals need fixing
  first (i.e. stage 1).
* **Re-run with another `CHANNEL`** before drawing conclusions about the
  method: weather's 21 columns range from smooth thermodynamic variables to
  near-binary ones (`rain`, `raining`), and the core is gauged per channel, so
  "the core learned nothing" on one channel is entirely compatible with strong
  structure on another. The cross-channel maximum printed in section 4 is the
  reminder of the flip side: nothing in this parameterization couples them.